# 00 · VisDrone-DET Data and Evaluation Protocol (SageMaker)

This notebook **does not train any models**. It performs four tasks:

1. Download the official VisDrone2019-DET train, val, and test-dev splits into the **current SageMaker JupyterLab space**. Verify that every image has a corresponding original annotation. Reruns neither re-download nor overwrite complete data.
2. Fix the official category IDs, reuse the prompts from Experiment 1, and define a proposed **7-base / 3-novel** split for VisDrone fine-tuning.
3. Save CSV/JSON manifests, archive SHA-256 checksums, and individual image/annotation SHA-256 checksums. Never overwrite changed v1 protocol files.
4. Create a Dataset in the new ClearML subproject **`SageMaker-Aerial-OVD-2026/VisDrone-DET-Protocol-Registry`**. Upload only the small protocol files. **Never upload images, original annotations, or ZIP archives to ClearML.**

Save this notebook in a **persistent SageMaker Studio JupyterLab space** before running it. The download URLs are the GitHub release mirrors used by the Ultralytics VisDrone configuration; these archives contain the original VisDrone images and TXT annotations, not Ultralytics-converted YOLO labels. If your university network cannot reach GitHub, ask your administrator for access or change the download source to an approved copy of exactly the same data version and create a new protocol version.

References: [VisDrone original repository](https://github.com/VisDrone/VisDrone-Dataset) · [Ultralytics VisDrone download configuration](https://docs.ultralytics.com/datasets/detect/visdrone/) · [ClearML Dataset API](https://clear.ml/docs/latest/docs/clearml_data/clearml_data_sdk/).

**Choose the novel categories before running the protocol-generation cell.** The 7/3 split below is a proposed research protocol, not an official VisDrone split. Here, *novel* means excluded from VisDrone fine-tuning, not necessarily unseen during the model's original pretraining. If you change categories or prompts, increment the local protocol version and the ClearML Dataset name/version; do not overwrite v1.

In [1]:
# Install dependencies in the active SageMaker kernel. Restart the kernel if package versions change.
%pip install -q "clearml>=1.16,<3" "requests>=2.31,<3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import hashlib
import io
import json
import os
import re
import shutil
import stat
import tempfile
import zipfile
from collections import Counter
from pathlib import Path, PurePosixPath

import requests

# Make sure this notebook runs from a persistent SageMaker JupyterLab space; Path.cwd() is its working directory.
ROOT = Path.cwd().resolve() / "SageMaker_VisDrone_OVD_2026"
DATA_ROOT = ROOT / "visdrone_det_original"     # Local images and original TXT annotations
ARCHIVE_ROOT = ROOT / "download_archives"      # Three original ZIP archives retained for verification
PROTOCOL_VERSION = "1.0.0"
PROTOCOL_DIR = ROOT / "protocol_v1"           # Small files to register in ClearML, and nothing else

# New ClearML subproject and Dataset name; do not reuse earlier Experiment 1 names.
CLEARML_DATASET_PROJECT = "SageMaker-Aerial-OVD-2026/VisDrone-DET-Protocol-Registry"
CLEARML_DATASET_NAME = "VisDrone-DET-fixed-splits-base7-novel3-v1"

# Mirror URL used by Ultralytics' VisDrone.yaml; omit test-challenge because labels are withheld.
ASSET_BASE = "https://github.com/ultralytics/assets/releases/download/v0.0.0"
SPLITS = {
    "train": ("VisDrone2019-DET-train", 6471),
    "val": ("VisDrone2019-DET-val", 548),
    "test": ("VisDrone2019-DET-test-dev", 1610),
}

# Original annotation category_id: 0=ignored region, 1...10=target classes, 11=others.
# Exclude IDs 0 and 11 from the ten-class evaluation.
CATEGORY_ID_TO_NAME = {
    1: "pedestrian", 2: "people", 3: "bicycle", 4: "car", 5: "van",
    6: "truck", 7: "tricycle", 8: "awning-tricycle", 9: "bus", 10: "motor",
}

# Reuse the default prompts from the earlier VisDrone / YOLO-World / Grounding DINO experiment.
# For example, people -> seated person is an experimental prompt, not an official category definition.
# Any later prompt ablation requires a new protocol version.
PROMPT_BY_CATEGORY = {
    "pedestrian": "pedestrian", "people": "seated person", "bicycle": "bicycle",
    "car": "car", "van": "van", "truck": "truck",
    "tricycle": "three-wheeled bicycle",
    "awning-tricycle": "covered three-wheeled vehicle",
    "bus": "bus", "motor": "motorcycle",
}

# Researcher-defined domain-held-out split. Decide on any changes before first publication.
BASE_NAMES = ("pedestrian", "people", "bicycle", "car", "van", "truck", "motor")
NOVEL_NAMES = ("tricycle", "awning-tricycle", "bus")
BASE_IDS = {category_id for category_id, name in CATEGORY_ID_TO_NAME.items() if name in BASE_NAMES}
NOVEL_IDS = {category_id for category_id, name in CATEGORY_ID_TO_NAME.items() if name in NOVEL_NAMES}

assert set(BASE_NAMES).isdisjoint(NOVEL_NAMES)
assert set(BASE_NAMES) | set(NOVEL_NAMES) == set(CATEGORY_ID_TO_NAME.values())
assert len(set(PROMPT_BY_CATEGORY.values())) == len(PROMPT_BY_CATEGORY)

for folder in (ROOT, DATA_ROOT, ARCHIVE_ROOT, PROTOCOL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage(ROOT).free / (1024 ** 3)
print("Workspace:", ROOT)
print("Disk free:", f"{free_gb:.2f} GiB")
print("ClearML Dataset project:", CLEARML_DATASET_PROJECT)
print("Base:", BASE_NAMES, "| Novel:", NOVEL_NAMES)
if free_gb < 5:
    print("WARNING: Downloading and extracting the three ZIP files may need over 4 GiB of free space. Check your space capacity.")

Workspace: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026
Disk free: 49.33 GiB
ClearML Dataset project: SageMaker-Aerial-OVD-2026/VisDrone-DET-Protocol-Registry
Base: ('pedestrian', 'people', 'bicycle', 'car', 'van', 'truck', 'motor') | Novel: ('tricycle', 'awning-tricycle', 'bus')


## 1. Download each official split once and verify the ZIP, images, and annotations

Before downloading, the notebook checks whether a **complete** official split already exists. It stops on partial data instead of silently overwriting anything. A downloaded ZIP is first saved as `.part`, validated, and then atomically renamed. Extraction accepts only regular files beneath the expected dataset path, preventing unexpected ZIP paths from escaping the destination. Archives remain in SageMaker for later verification.

If GitHub is inaccessible, do not silently switch to a third-party dataset with different preprocessing or annotations. Do not substitute Ultralytics-generated YOLO labels for the original TXT annotations.

In [3]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def validate_split(split: str, folder: Path) -> list[Path]:
    _, expected_count = SPLITS[split]
    image_dir, annotation_dir = folder / "images", folder / "annotations"
    if not image_dir.is_dir() or not annotation_dir.is_dir():
        raise RuntimeError(f"{folder}: missing images/ or annotations/; check whether extraction finished.")
    images = sorted(image_dir.glob("*.jpg"))
    annotations = list(annotation_dir.glob("*.txt"))
    if len(images) != expected_count or len(annotations) != expected_count:
        raise RuntimeError(
            f"{folder}: image/annotation counts={len(images)}/{len(annotations)}; "
            f"expected {expected_count} each. Incomplete data will not be overwritten or ignored."
        )
    image_ids = {path.stem for path in images}
    annotation_ids = {path.stem for path in annotations}
    if image_ids != annotation_ids:
        raise RuntimeError(f"{folder}: image IDs do not exactly match annotation IDs.")
    return images


def download_zip(url: str, zip_path: Path) -> None:
    if zip_path.exists():
        if not zipfile.is_zipfile(zip_path):
            raise RuntimeError(f"Existing ZIP is corrupt: {zip_path}. Inspect it manually; no automatic overwrite.")
        print("Using existing ZIP:", zip_path.name)
        return
    part_path = zip_path.with_suffix(zip_path.suffix + ".part")
    if part_path.exists():
        raise RuntimeError(f"Unfinished download found at {part_path}. Inspect and remove it manually before retrying.")
    print("Downloading:", url)
    try:
        with requests.get(url, stream=True, timeout=(30, 120)) as response:
            response.raise_for_status()
            with part_path.open("xb") as handle:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        handle.write(chunk)
        if not zipfile.is_zipfile(part_path):
            raise RuntimeError("Downloaded file is not a valid ZIP; it may be a login page, error page, or incomplete download.")
        os.replace(part_path, zip_path)
    except Exception:
        # Keep .part for troubleshooting; never treat a partial download as valid data.
        raise


def extract_checked(zip_path: Path, split: str, destination: Path) -> None:
    expected_folder = SPLITS[split][0]
    with tempfile.TemporaryDirectory(prefix=f"extract_{split}_", dir=DATA_ROOT) as temp_name:
        temp_root = Path(temp_name)
        with zipfile.ZipFile(zip_path) as archive:
            for member in archive.infolist():
                rel = PurePosixPath(member.filename)
                if (not rel.parts or rel.is_absolute() or ".." in rel.parts
                        or rel.parts[0] != expected_folder
                        or stat.S_ISLNK(member.external_attr >> 16)):
                    raise RuntimeError(f"ZIP contains an unsafe or unexpected path: {member.filename!r}")
            archive.extractall(temp_root)
        extracted = temp_root / expected_folder
        validate_split(split, extracted)
        if destination.exists():
            raise RuntimeError(f"Destination already exists and will not be overwritten: {destination}")
        shutil.move(str(extracted), str(destination))


archive_hashes = {}
for split, (archive_stem, expected) in SPLITS.items():
    destination = DATA_ROOT / archive_stem
    archive_path = ARCHIVE_ROOT / f"{archive_stem}.zip"
    download_zip(f"{ASSET_BASE}/{archive_stem}.zip", archive_path)
    archive_hashes[split] = sha256_file(archive_path)
    if destination.exists():
        validate_split(split, destination)
    else:
        extract_checked(archive_path, split, destination)
    print(f"{split:5} {expected:5d} images | archive sha256: {archive_hashes[split][:16]}...")

print("All three official train/val/test-dev splits validated.")

Downloading: https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip
train  6471 images | archive sha256: 86a77eba93137bfc...
Downloading: https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip
val     548 images | archive sha256: abeea063037e5d20...
Downloading: https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-test-dev.zip
test   1610 images | archive sha256: b28a94b06dfd9e36...
All three official train/val/test-dev splits validated.


## 2. Count the classes and generate a non-overwritable v1 protocol

`train_base_safe.csv` contains **only training images without any annotated novel object**. This avoids the common mistake of deleting novel-object labels while leaving their objects in training images as apparent background. Later training notebooks must create model-specific base-only labels themselves and must never use novel-object annotations for fine-tuning.

Use val for development and reserve test-dev for final reporting. Do not repeatedly select prompts, thresholds, or splits based on test-dev performance. *Novel* here means held out from VisDrone fine-tuning; it does not mean the pretrained model has never encountered the object type.

In [4]:
def parse_visdrone_annotation(annotation_path: Path) -> Counter:
    counts = Counter()
    with annotation_path.open("r", encoding="utf-8-sig") as handle:
        for line_number, raw in enumerate(handle, 1):
            if not raw.strip():
                continue
            fields = [field.strip() for field in raw.split(",")]
            if len(fields) < 6:
                raise ValueError(f"{annotation_path}:{line_number}: fewer than six fields")
            try:
                category_id = int(fields[5])
            except ValueError as exc:
                raise ValueError(f"{annotation_path}:{line_number}: category ID is not an integer") from exc
            if category_id not in {0, 11, *CATEGORY_ID_TO_NAME}:
                raise ValueError(f"{annotation_path}:{line_number}: unknown category ID {category_id}")
            counts[category_id] += 1
    return counts


def csv_text(rows: list[dict], fieldnames: list[str]) -> str:
    buffer = io.StringIO(newline="")
    writer = csv.DictWriter(buffer, fieldnames=fieldnames, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    return buffer.getvalue()


def pretty_json(value: dict) -> str:
    return json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True) + "\n"


manifest_cols = [
    "split", "image_id", "image_relpath", "annotation_relpath",
    "image_sha256", "annotation_sha256", "num_base_objects", "num_novel_objects",
]
all_rows = {}
class_tallies = {}
for split, (archive_stem, _) in SPLITS.items():
    images = validate_split(split, DATA_ROOT / archive_stem)
    rows, class_counts = [], Counter()
    for image in images:
        annotation = image.parent.parent / "annotations" / (image.stem + ".txt")
        object_counts = parse_visdrone_annotation(annotation)
        class_counts.update(object_counts)
        rows.append({
            "split": split,
            "image_id": image.stem,
            "image_relpath": image.relative_to(DATA_ROOT).as_posix(),
            "annotation_relpath": annotation.relative_to(DATA_ROOT).as_posix(),
            "image_sha256": sha256_file(image),
            "annotation_sha256": sha256_file(annotation),
            "num_base_objects": sum(object_counts[k] for k in BASE_IDS),
            "num_novel_objects": sum(object_counts[k] for k in NOVEL_IDS),
        })
    all_rows[split], class_tallies[split] = rows, class_counts
    print(split, "images:", len(rows), "objects:", sum(class_counts.values()))

base_safe_rows = [row for row in all_rows["train"] if row["num_novel_objects"] == 0]
print("Base-only eligible training images (no annotated novel object):", len(base_safe_rows))
if not base_safe_rows:
    raise RuntimeError("No base-only training images remain. Redesign the base/novel split before publishing v1.")

category_spec = {
    "official_category_id_to_name": {str(k): v for k, v in CATEGORY_ID_TO_NAME.items()},
    "excluded_official_ids": {"0": "ignored region", "11": "others"},
    "base_official_ids": sorted(BASE_IDS),
    "novel_official_ids": sorted(NOVEL_IDS),
    "base_category_names": list(BASE_NAMES),
    "novel_category_names": list(NOVEL_NAMES),
    "prompt_by_category_name": PROMPT_BY_CATEGORY,
    "novel_definition": "Categories excluded from VisDrone fine-tuning; NOT necessarily unseen in model pretraining.",
}

protocol_spec = {
    "version": PROTOCOL_VERSION,
    "source": "VisDrone2019-DET original images and annotations, Ultralytics assets mirror",
    "original_repo": "https://github.com/VisDrone/VisDrone-Dataset",
    "mirror_base": ASSET_BASE,
    "split_to_archive": {split: f"{name}.zip" for split, (name, _) in SPLITS.items()},
    "archive_sha256": archive_hashes,
    "expected_images": {split: count for split, (_, count) in SPLITS.items()},
    "manifest_paths": {split: f"{split}.csv" for split in SPLITS},
    "base_only_train_manifest": "train_base_safe.csv",
    "data_path_rule": "All image_relpath and annotation_relpath fields are relative to DATA_ROOT.",
    "evaluation_rule": "Use fixed val during development, hold test-dev for final evaluation.",
    "training_rule": "Train on base-safe images only; retain original official IDs in the protocol, remap in training notebooks as needed.",
}

summary = {
    "images_by_split": {split: len(rows) for split, rows in all_rows.items()},
    "objects_per_official_category_by_split": {
        split: {str(k): int(tallies[k]) for k in range(12)}
        for split, tallies in class_tallies.items()
    },
    "train_base_safe_images": len(base_safe_rows),
}

# Only files in PROTOCOL_DIR enter the ClearML Dataset: no JPGs, raw annotations, weights, or archives.
protocol_files = {
    "categories_and_prompts.json": pretty_json(category_spec),
    "protocol.json": pretty_json(protocol_spec),
    "summary.json": pretty_json(summary),
    "train_base_safe.csv": csv_text(base_safe_rows, manifest_cols),
}
for split, rows in all_rows.items():
    protocol_files[f"{split}.csv"] = csv_text(rows, manifest_cols)

fingerprint = hashlib.sha256()
for name, content in sorted(protocol_files.items()):
    fingerprint.update(name.encode("utf-8") + b"\x00" + content.encode("utf-8") + b"\x00")
protocol_fingerprint = fingerprint.hexdigest()
protocol_files["protocol_fingerprint.txt"] = protocol_fingerprint + "\n"

expected_names = set(protocol_files)
unexpected = {p.name for p in PROTOCOL_DIR.iterdir() if p.is_file()} - expected_names
if unexpected:
    raise RuntimeError(f"Unexpected files in the protocol directory: {unexpected}. Inspect them before registering with ClearML.")
for filename, content in protocol_files.items():
    target = PROTOCOL_DIR / filename
    if target.exists() and target.read_text(encoding="utf-8") != content:
        raise RuntimeError(f"Frozen v1 protocol changed: {target}. Do not overwrite it; use a new v2 directory/version/name.")
for filename, content in protocol_files.items():
    target = PROTOCOL_DIR / filename
    if not target.exists():
        target.write_text(content, encoding="utf-8")

print("Protocol dir:", PROTOCOL_DIR)
print("Protocol fingerprint:", protocol_fingerprint)
print("Lightweight files to register:")
for file in sorted(PROTOCOL_DIR.iterdir()):
    print(f"  {file.name:32s} {file.stat().st_size / 1024:.1f} KiB")
print("All other files (images, original annotations, ZIPs) remain only in SageMaker.")

train images: 6471 objects: 353550
val images: 548 objects: 40169
test images: 1610 objects: 77547
Base-only eligible training images (no annotated novel object): 2695
Protocol dir: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/protocol_v1
Protocol fingerprint: 158cb2c904016fa324c236b998393b1a94c176a4eabedb159c76bdeffc66c195
Lightweight files to register:
  categories_and_prompts.json      1.1 KiB
  protocol.json                    1.2 KiB
  protocol_fingerprint.txt         0.1 KiB
  summary.json                     0.8 KiB
  test.csv                         457.7 KiB
  train.csv                        1808.0 KiB
  train_base_safe.csv              752.7 KiB
  val.csv                          150.1 KiB
All other files (images, original annotations, ZIPs) remain only in SageMaker.


## 3. Review the class counts and register a new ClearML Dataset

If the 7/3 split does not meet your research needs, stop and update `BASE_NAMES` / `NOVEL_NAMES` **before generating v1 protocol files**. After ClearML finalization, keep v1 unchanged; for a different protocol, use `protocol_v2`, a different Dataset name, and an incremented `PROTOCOL_VERSION`.

This cell uploads only the eight small files from `PROTOCOL_DIR` to ClearML, after checking their total size. It reuses a completed Dataset with the same name and fingerprint. If an existing Dataset has a different fingerprint or an incomplete upload, it stops rather than creating a duplicate. Configure ClearML credentials in SageMaker Secrets, environment variables, or `clearml.conf` first; **never put API keys in the notebook**.

In [6]:
import os
from getpass import getpass

os.environ["CLEARML_WEB_HOST"] = "https://app.clear.ml"
os.environ["CLEARML_API_HOST"] = "https://api.clear.ml"
os.environ["CLEARML_FILES_HOST"] = "https://files.clear.ml"

os.environ["CLEARML_API_ACCESS_KEY"] = getpass("ClearML access key: ")
os.environ["CLEARML_API_SECRET_KEY"] = getpass("ClearML secret key: ")

print("ClearML credentials set for this kernel.")

ClearML access key:  ········
ClearML secret key:  ········


ClearML credentials set for this kernel.


In [7]:
from clearml import Dataset, Task

# Dataset.create creates its own ClearML task; do not leave another Task.init active in this kernel.
if Task.current_task() is not None:
    raise RuntimeError("A ClearML Task is already active in this kernel. Run this data-protocol notebook in a fresh kernel.")

allowed = set(protocol_files)
local_files = [p for p in PROTOCOL_DIR.iterdir() if p.is_file()]
assert {p.name for p in local_files} == allowed
assert not any(p.suffix.lower() in {".jpg", ".jpeg", ".png", ".zip", ".pt"} for p in local_files)
total_protocol_bytes = sum(p.stat().st_size for p in local_files)
if total_protocol_bytes >= 20 * 1024 * 1024:
    raise RuntimeError("Protocol size is >=20 MiB; only small files may be uploaded. Upload aborted.")

records = Dataset.list_datasets(
    dataset_project=CLEARML_DATASET_PROJECT,
    partial_name=re.escape(CLEARML_DATASET_NAME),
    only_completed=False,
    recursive_project_search=False,
)
same_name = [record for record in records if record.get("name") == CLEARML_DATASET_NAME]
if len(same_name) > 1:
    raise RuntimeError("Multiple ClearML Datasets have this name. Confirm the correct Dataset ID manually before continuing.")

if same_name:
    ds = Dataset.get(dataset_id=same_name[0]["id"])
    if not ds.is_final():
        raise RuntimeError(f"An existing Dataset with the same name is not finalized (id={ds.id}). Check its status in ClearML.")
    existing_metadata = ds.get_metadata() or {}
    if existing_metadata.get("protocol_fingerprint") != protocol_fingerprint:
        raise RuntimeError("An existing ClearML Dataset has the same name but a different fingerprint. Create v2 instead of reusing or overwriting v1.")
    print("Reusing finalized ClearML Dataset:", ds.id)
else:
    ds = Dataset.create(
        dataset_project=CLEARML_DATASET_PROJECT,
        dataset_name=CLEARML_DATASET_NAME,
        dataset_version=PROTOCOL_VERSION,
        description="Only VisDrone DET split manifests, SHA-256s, prompts and base/novel protocol; NO raw images or annotations.",
    )
    ds.add_files(path=str(PROTOCOL_DIR))  # The directory contains only the eight approved small files.
    ds.set_metadata({
        "protocol_fingerprint": protocol_fingerprint,
        "split_counts": summary["images_by_split"],
        "train_base_safe_images": len(base_safe_rows),
        "raw_images_uploaded": False,
    })
    ds.upload()  # Only these small files upload to the default ClearML file server.
    ds.finalize()
    print("Created new finalized ClearML Dataset:", ds.id)

id_file = ROOT / "clearml_protocol_dataset_id.txt"  # Kept in SageMaker; not included in the ClearML Dataset.
if id_file.exists() and id_file.read_text(encoding="utf-8").strip() != ds.id:
    raise RuntimeError("Local Dataset ID conflicts with the server ID. Check both IDs manually; do not overwrite the old ID.")
id_file.write_text(ds.id + "\n", encoding="utf-8")

print("ClearML Dataset ID:", ds.id)
print("Project:", CLEARML_DATASET_PROJECT)
print("Uploaded protocol only:", total_protocol_bytes / (1024 ** 2), "MiB")
print("Local raw image root:", DATA_ROOT)
print("Next notebooks: read this Dataset ID and protocol manifests; never re-randomize official splits.")

2026-09-13 16:04:05,020 - clearml - WARNING - No projects were found with name(s): ^SageMaker\-Aerial\-OVD\-2026/VisDrone\-DET\-Protocol\-Registry$, ^SageMaker\-Aerial\-OVD\-2026/VisDrone\-DET\-Protocol\-Registry/\.datasets/.*
ClearML results page: https://app.clear.ml/projects/76697fe2d0154de1a54f1748fe4119a3/tasks/738dce9e1a5440dab34195418a997a20/output/log
ClearML dataset page: https://app.clear.ml/datasets/simple/76697fe2d0154de1a54f1748fe4119a3/experiments/738dce9e1a5440dab34195418a997a20
Generating SHA2 hash for 8 files


100%|██████████| 8/8 [00:00<00:00, 1201.72it/s]

Hash generation completed


Displaying metadata in the UI is only supported for pandas Dataframes for now. Skipping!
Uploading dataset changes (8 files compressed to 1.91 MiB) to https://files.clear.ml
File compression and upload completed: total size 1.91 MiB, 1 chunk(s) stored (average size 1.91 MiB)
Created new finalized ClearML Dataset: 738dce9e1a5440dab34195418a997a20
ClearML Dataset ID: 738dce9e1a5440dab34195418a997a20
Project: SageMaker-Aerial-OVD-2026/VisDrone-DET-Protocol-Registry
Uploaded protocol only: 3.0972232818603516 MiB
Local raw image root: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/visdrone_det_original
Next notebooks: read this Dataset ID and protocol manifests; never re-randomize official splits.
